###Eventhub Ingestion

In [0]:
ADLS_BASE = "abfss://cryptoblob@cryptoanalysisadls.dfs.core.windows.net"
BRONZE_PATH = f"{ADLS_BASE}/bronze/"

bronze_df = spark.read.format("delta").load(BRONZE_PATH)

In [0]:
from pyspark.sql.functions import expr, when, to_timestamp, current_timestamp, to_date

bronze_df = bronze_df \
    .withColumn("event_time", to_timestamp("last_updated")) \
    .withColumn("ingestion_time", current_timestamp()) \
    .withColumn("ingestion_date", to_date("ingestion_time"))

In [0]:
upsert_to_bronze(bronze_df.dropDuplicates(['id']))

In [0]:
# Validate — Preview Bronze Table (null-aware)

df_bronze_check = spark.read.format("delta").load(BRONZE_PATH)

total_rows  = df_bronze_check.count()
null_rows   = df_bronze_check.filter("id IS NULL").count()
valid_rows  = total_rows - null_rows

print(f"Total rows in Bronze : {total_rows}")
print(f"Valid rows (id != null): {valid_rows}")
print(f"Null/corrupt rows    : {null_rows}")
print(f"Columns              : {df_bronze_check.columns}")

if null_rows > 0:
    print(
        f"\n⚠️  WARNING: {null_rows} null rows detected. "
        "These are likely from old schema-mismatched runs. "
        "Run the ONE-TIME REPAIR cell below to clean the Bronze table."
    )

print("\nSample valid rows:")
display(
    df_bronze_check
    .filter("id IS NOT NULL")   # only show valid rows in preview
    .select(
        "id", "name", "symbol",
        "current_price", "market_cap",
        "total_volume", "price_change_percentage_24h",
        "last_updated", "ingestion_time", "ingestion_date"
    )
    .limit(10)
)

Total rows in Bronze : 14998
Valid rows (id != null): 14998
Null/corrupt rows    : 0
Columns              : ['kafka_key', 'topic', 'partition', 'offset', 'event_hub_enqueue_ts', 'ingest_ts', 'id', 'symbol', 'name', 'current_price', 'market_cap', 'market_cap_rank', 'fully_diluted_valuation', 'total_volume', 'high_24h', 'low_24h', 'price_change_24h', 'price_change_percentage_24h', 'market_cap_change_24h', 'market_cap_change_percentage_24h', 'circulating_supply', 'total_supply', 'max_supply', 'ath', 'ath_change_percentage', 'ath_date', 'atl', 'atl_change_percentage', 'atl_date', 'roi', 'last_updated', 'event_time', 'ingestion_time', 'ingestion_date']

Sample valid rows:


id,name,symbol,current_price,market_cap,total_volume,price_change_percentage_24h,last_updated,ingestion_time,ingestion_date
-5,🟥🟪🟦🟩🟨🟧,🟥🟩,4.3509E-4,434869.0,56232.0,2.81417,2026-04-21T07:59:10.910Z,2026-04-24T09:18:44.719818Z,2026-04-24
-6,""" """,""" """,8.2574E-4,522393.0,287.39,0.8519,2026-04-21T07:59:11.400Z,2026-04-24T09:18:44.719818Z,2026-04-24
-7,Voidify,∅,7.4371E-4,713669.0,4617.33,-1.71392,2026-04-21T07:59:11.562Z,2026-04-24T09:18:44.719818Z,2026-04-24
0x,0x Protocol,zrx,0.111466,9.4579273E7,9432639.0,4.41839,2026-04-21T07:52:56.113Z,2026-04-24T09:18:44.719818Z,2026-04-24
0x0-ai-ai-smart-contract,0x0,0x0,0.00584523,5212467.0,9235.96,-1.50701,2026-04-21T07:55:21.866Z,2026-04-24T09:18:44.719818Z,2026-04-24
0xgasless-2,0xGasless,0xgas,0.058254,641154.0,246.28,1.83903,2026-04-21T07:59:16.004Z,2026-04-24T09:18:44.719818Z,2026-04-24
0xgen,0xGen,xgn,3.2705E-4,228962.0,1600.03,-4.61507,2026-04-21T07:59:50.388Z,2026-04-24T09:18:44.719818Z,2026-04-24
0xmonk,0xMonk by Virtuals,monk,2.1531E-4,215308.0,2.08,-1.24453,2026-04-21T07:02:25.208Z,2026-04-24T09:18:44.719818Z,2026-04-24
1-coin-can-change-your-life,1 Coin Can Change Your Life,1-coin-can-change-your-life,0.00113732,1137141.0,220043.0,-2.00506,2026-04-21T07:58:23.995Z,2026-04-24T09:18:44.719818Z,2026-04-24
1000x-by-virtuals,1000x by Virtuals,1000x,0.00101012,1003307.0,38610.0,10.84013,2026-04-21T07:58:33.811Z,2026-04-24T09:18:44.719818Z,2026-04-24
